# 02. Пайплайн кандидатогенерации

В этом блокноте строится локальная валидация и семантический retrieval.


## 1. Окружение и данные


In [ ]:
import gc
import os
import sys
from pathlib import Path

import faiss
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, save_npz
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.geo import build_location_centers, haversine_distance
from src.text_features import (
    build_lexical_item_text,
    build_query_text,
    build_semantic_item_text,
)
from src.retrieval import (
    build_tfidf_index,
    encode_items,
    encode_queries,
    load_semantic_model,
)


In [ ]:
DATA_DIR = PROJECT_ROOT / "dataset"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)


In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
benchmark_queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
benchmark_items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

In [ ]:
train.shape, benchmark_queries.shape, benchmark_items.shape

((497673, 19), (2452, 6), (189212, 14))

## 2. Локальная валидация




In [ ]:
SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

In [58]:
# Полный поисковый контекст не должен одновременно попасть в обе части.
query_group_ids = train.groupby(SEARCH_COLUMNS, dropna=False).ngroup()
unique_group_ids = query_group_ids.unique()

development_group_ids, validation_group_ids = train_test_split(
    unique_group_ids,
    test_size=0.2,
    random_state=42,
)

development_train = train[
    query_group_ids.isin(development_group_ids)
].copy()

validation_rows = train[
    query_group_ids.isin(validation_group_ids)
].copy()


KeyboardInterrupt: 

In [ ]:
development_train.shape, validation_rows.shape

((398757, 19), (98916, 19))

In [ ]:
set(development_group_ids) & set(validation_group_ids)

set()

Каталог содержит по одной строке на `item_id`. Для локальной проверки в нём остаются
все объявления train, включая релевантные validation-объекты: это имитирует условие
задачи, где искомые объявления присутствуют в доступном корпусе.


In [ ]:
# Создание каталога объявлений
ITEM_COLUMNS = [
    "item_id",
    "item_title_raw",
    "item_rating_reviews_count",
    "item_rating",
    "item_price",
    "item_microcat_id",
    "item_longitude",
    "item_location_id",
    "item_latitude",
    "item_is_phone_hidden",
    "item_is_message_forbidden",
    "item_infm_params_text",
    "item_description_raw",
    "item_category_id",
]

# На каждый айтем должна остаться одна строка
candidate_items = (
    train[ITEM_COLUMNS]
    .drop_duplicates("item_id", keep="last")
    .reset_index(drop=True)
)

In [ ]:
candidate_items.shape

(344825, 14)

In [ ]:
validation_queries = (
    validation_rows
    .drop_duplicates(SEARCH_COLUMNS + ["item_id"])
    .groupby(SEARCH_COLUMNS, dropna=False)
    .agg(relevant_item_ids=("item_id", list))
    .reset_index()
)

validation_queries.shape

(70893, 6)

In [ ]:
# Проверим какие из запросов встречались в тренировочной выборке
development_query_texts = set(
    development_train["search_query"]
)

validation_queries["query_text_seen"] = (
    validation_queries["search_query"]
    .isin(development_query_texts)
)
validation_queries["query_text_seen"].value_counts()

query_text_seen
True     60532
False    10361
Name: count, dtype: int64

Итого получено 70 893 validation-контекста: 60 532 с текстом, встречавшимся в
development, и 10 361 с новым текстом. Полная комбинация признаков запроса при этом
в development не повторяется.


## 3. Подготовка признаков


### 3.1 Географическая привязка запроса

Для `search_location_id` берём медианный центр доступных объявлений. Если центр не
найден, используем только development-взаимодействия как fallback.


In [ ]:
# Координаты локаций восстанавливаем по всем доступным объявлениям без разметки.
location_reference_items = pd.concat(
    [
        train[["item_location_id", "item_latitude", "item_longitude"]],
        benchmark_items[["item_location_id", "item_latitude", "item_longitude"]],
    ],
    ignore_index=True,
).drop_duplicates()

location_centers = build_location_centers(location_reference_items)

query_location_centers = (
    location_centers
    .reset_index()
    .rename(
        columns={
            "item_location_id": "search_location_id",
            "latitude": "query_latitude",
            "longitude": "query_longitude",
        }
    )[["search_location_id", "query_latitude", "query_longitude"]]
)

# Fallback использует только development-часть, чтобы не подглядывать в validation-связи.
fallback_location_centers = (
    development_train
    .dropna(
        subset=["search_location_id", "item_latitude", "item_longitude"]
    )
    .groupby("search_location_id", as_index=False)
    .agg(
        fallback_latitude=("item_latitude", "median"),
        fallback_longitude=("item_longitude", "median"),
    )
)

validation_queries_with_geo = (
    validation_queries
    .merge(
        query_location_centers,
        on="search_location_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        fallback_location_centers,
        on="search_location_id",
        how="left",
        validate="many_to_one",
    )
)

validation_queries_with_geo["query_latitude"] = (
    validation_queries_with_geo["query_latitude"]
    .fillna(validation_queries_with_geo["fallback_latitude"])
)
validation_queries_with_geo["query_longitude"] = (
    validation_queries_with_geo["query_longitude"]
    .fillna(validation_queries_with_geo["fallback_longitude"])
)
validation_queries_with_geo = validation_queries_with_geo.drop(
    columns=["fallback_latitude", "fallback_longitude"]
)


In [ ]:
item_location_centers = (
    location_centers
    .reset_index()
    .rename(
        columns={
            "latitude": "location_latitude",
            "longitude": "location_longitude",
        }
    )
)

candidate_items_with_geo = candidate_items.merge(
    item_location_centers,
    on="item_location_id",
    how="left",
    validate="many_to_one",
)

assert np.array_equal(
    candidate_items_with_geo["item_id"].to_numpy(),
    candidate_items["item_id"].to_numpy(),
)

In [ ]:
item_latitudes = (
    pd.to_numeric(
        candidate_items_with_geo["item_latitude"],
        errors="coerce",
    )
    .fillna(
        candidate_items_with_geo["location_latitude"]
    )
    .to_numpy(dtype=np.float32)
)

item_longitudes = (
    pd.to_numeric(
        candidate_items_with_geo["item_longitude"],
        errors="coerce",
    )
    .fillna(
        candidate_items_with_geo["location_longitude"]
    )
    .to_numpy(dtype=np.float32)
)

query_latitudes = pd.to_numeric(
    validation_queries_with_geo["query_latitude"],
    errors="coerce",
).to_numpy(dtype=np.float32)

query_longitudes = pd.to_numeric(
    validation_queries_with_geo["query_longitude"],
    errors="coerce",
).to_numpy(dtype=np.float32)

In [ ]:
np.save(
    ARTIFACTS_DIR / "candidate_latitudes.npy",
    item_latitudes,
)

np.save(
    ARTIFACTS_DIR / "candidate_longitudes.npy",
    item_longitudes,
)

np.save(
    ARTIFACTS_DIR / "validation_query_latitudes.npy",
    query_latitudes,
)

np.save(
    ARTIFACTS_DIR / "validation_query_longitudes.npy",
    query_longitudes,
)

In [ ]:
item_coordinate_coverage = np.mean(
    np.isfinite(item_latitudes)
    & np.isfinite(item_longitudes)
)

query_coordinate_coverage = np.mean(
    np.isfinite(query_latitudes)
    & np.isfinite(query_longitudes)
)

item_coordinate_coverage, query_coordinate_coverage

(np.float64(1.0), np.float64(0.9996050385792673))

In [ ]:
(
    item_latitudes.shape,
    query_latitudes.shape,
)

((344825,), (70893,))

In [ ]:
validation_queries_with_geo[
    ["query_latitude", "query_longitude"]
].isna().mean()

query_latitude     0.000395
query_longitude    0.000395
dtype: float64

### 3.2 Текстовые представления

- Запрос: `search_query + search_infm_params_text`.
- TF-IDF-документ: заголовок + параметры объявления.
- E5-документ: заголовок + параметры + первые 500 символов описания.


In [ ]:
query_texts = build_query_text(validation_queries_with_geo)
lexical_item_texts = build_lexical_item_text(candidate_items)
semantic_item_texts = build_semantic_item_text(candidate_items)

len(query_texts), len(lexical_item_texts), len(semantic_item_texts)

(70893, 344825, 344825)

## 4. Лексический канал TF-IDF

Матрица и словарь уже сохранены на диск. На следующем этапе этот канал будет искать
точные локальные и глобальные совпадения и добавлять разнообразие к E5-кандидатам.


In [ ]:
TFIDF_VECTORIZER_PATH = ARTIFACTS_DIR / "tfidf_vectorizer.joblib"
TFIDF_ITEM_MATRIX_PATH = ARTIFACTS_DIR / "tfidf_item_matrix.npz"

if TFIDF_VECTORIZER_PATH.exists() and TFIDF_ITEM_MATRIX_PATH.exists():
    tfidf_vectorizer = joblib.load(TFIDF_VECTORIZER_PATH)
    tfidf_item_matrix = load_npz(TFIDF_ITEM_MATRIX_PATH)
else:
    tfidf_vectorizer, tfidf_item_matrix, _ = build_tfidf_index(
        lexical_item_texts
    )
    joblib.dump(tfidf_vectorizer, TFIDF_VECTORIZER_PATH)
    save_npz(TFIDF_ITEM_MATRIX_PATH, tfidf_item_matrix)

tfidf_query_matrix = tfidf_vectorizer.transform(query_texts)


In [ ]:
tfidf_item_matrix.shape, tfidf_query_matrix.shape


((344825, 172789), (70893, 172789))

In [ ]:
from scipy.sparse import save_npz

TFIDF_QUERY_MATRIX_PATH = (
    ARTIFACTS_DIR / "tfidf_validation_query_matrix.npz"
)

save_npz(
    TFIDF_QUERY_MATRIX_PATH,
    tfidf_query_matrix,
)

## 5. Семантические эмбеддинги E5

`intfloat/multilingual-e5-small` используется как готовый энкодер, без дообучения.
Нормализация включена в `encode_items` и `encode_queries`, поэтому inner product
совпадает с cosine similarity.


In [ ]:
# Модель загружаем только если какого-либо кэша эмбеддингов ещё нет.
os.environ.pop("SSLKEYLOGFILE", None)

ITEM_EMBEDDINGS_PATH = ARTIFACTS_DIR / "local_item_embeddings_e5_small.npy"
QUERY_EMBEDDINGS_PATH = (
    ARTIFACTS_DIR / "local_validation_query_embeddings_e5_small.npy"
)

semantic_model = None
if not ITEM_EMBEDDINGS_PATH.exists() or not QUERY_EMBEDDINGS_PATH.exists():
    semantic_model = load_semantic_model()
    semantic_model.max_seq_length = 256


In [ ]:
if ITEM_EMBEDDINGS_PATH.exists():
    item_embeddings = np.load(ITEM_EMBEDDINGS_PATH, mmap_mode="r")
else:
    item_embeddings = encode_items(
        model=semantic_model,
        item_texts=semantic_item_texts,
        batch_size=64,
    )
    np.save(ITEM_EMBEDDINGS_PATH, item_embeddings)


In [ ]:
if QUERY_EMBEDDINGS_PATH.exists():
    query_embeddings = np.load(QUERY_EMBEDDINGS_PATH, mmap_mode="r")
else:
    query_embeddings = encode_queries(
        model=semantic_model,
        query_texts=query_texts,
        batch_size=128,
    )
    np.save(QUERY_EMBEDDINGS_PATH, query_embeddings)


In [ ]:
query_embeddings.shape, query_embeddings.dtype

((70893, 384), dtype('float32'))

In [ ]:
np.linalg.norm(query_embeddings[:5], axis=1)

array([1.       , 1.       , 1.       , 1.       , 1.0000001],
      dtype=float32)

Сохранённые формы: 344 825 объявлений и 70 893 запроса, размерность эмбеддинга 384.
Эмбеддинги объявлений занимают около 505 MB, запросов — около 104 MB.


## 6. Глобальный семантический поиск


Для всего корпуса используется приближённый HNSW-индекс FAISS (`M=16`). Он строится
один раз и затем загружается с диска.


In [ ]:
HNSW_INDEX_PATH = ARTIFACTS_DIR / "local_semantic_hnsw_m16_e5_small.faiss"
GLOBAL_INDICES_PATH = ARTIFACTS_DIR / "local_semantic_top100_indices.npy"
GLOBAL_SCORES_PATH = ARTIFACTS_DIR / "local_semantic_top100_scores.npy"

# Если результаты поиска уже есть, сам индекс не нужен для оценки и не занимает RAM.
semantic_index = None

if not GLOBAL_INDICES_PATH.exists() or not GLOBAL_SCORES_PATH.exists():
    faiss.omp_set_num_threads(4)

    if HNSW_INDEX_PATH.exists():
        semantic_index = faiss.read_index(str(HNSW_INDEX_PATH))
    else:
        semantic_index = faiss.IndexHNSWFlat(
            item_embeddings.shape[1],
            16,
            faiss.METRIC_INNER_PRODUCT,
        )
        semantic_index.hnsw.efConstruction = 80

        for start in range(0, len(item_embeddings), 5_000):
            end = min(start + 5_000, len(item_embeddings))
            semantic_index.add(
                np.ascontiguousarray(
                    item_embeddings[start:end],
                    dtype=np.float32,
                )
            )

        faiss.write_index(semantic_index, str(HNSW_INDEX_PATH))


In [ ]:
if semantic_index is None:
    print("Глобальные top-100 уже сохранены; HNSW не загружался в память.")
else:
    print("Объектов в HNSW:", semantic_index.ntotal)


Глобальные top-100 уже сохранены; HNSW не загружался в память.


In [ ]:
GLOBAL_INDICES_PATH = ARTIFACTS_DIR / "local_semantic_top100_indices.npy"
GLOBAL_SCORES_PATH = ARTIFACTS_DIR / "local_semantic_top100_scores.npy"

if GLOBAL_INDICES_PATH.exists() and GLOBAL_SCORES_PATH.exists():
    semantic_indices = np.load(GLOBAL_INDICES_PATH, mmap_mode="r")
    semantic_scores = np.load(GLOBAL_SCORES_PATH, mmap_mode="r")
else:
    semantic_index.hnsw.efSearch = 128
    semantic_indices = np.empty((len(query_embeddings), 100), dtype=np.int64)
    semantic_scores = np.empty((len(query_embeddings), 100), dtype=np.float32)

    for start in range(0, len(query_embeddings), 1_000):
        end = min(start + 1_000, len(query_embeddings))
        scores, indices = semantic_index.search(
            np.ascontiguousarray(
                query_embeddings[start:end],
                dtype=np.float32,
            ),
            100,
        )
        semantic_indices[start:end] = indices
        semantic_scores[start:end] = scores

    np.save(GLOBAL_INDICES_PATH, semantic_indices)
    np.save(GLOBAL_SCORES_PATH, semantic_scores)


In [ ]:
semantic_indices.shape, semantic_scores.shape

((70893, 100), (70893, 100))

In [ ]:
semantic_scores[0, :10]

memmap([0.8785615 , 0.87477565, 0.8736166 , 0.87349737, 0.872005  ,
        0.8718761 , 0.8718244 , 0.87058496, 0.8696697 , 0.86904997],
       dtype=float32)

### 6.1 Оценка глобального E5


In [ ]:
# FAISS возвращает позиции строк; сохраняем стабильное соответствие position -> item_id.
candidate_item_ids = np.asarray(
    candidate_items["item_id"].to_numpy(),
    dtype="U16",
)
candidate_location_ids = (
    candidate_items["item_location_id"].fillna(-1).astype(np.int64).to_numpy()
)
candidate_category_ids = (
    candidate_items["item_category_id"].fillna(-1).astype(np.int64).to_numpy()
)
validation_location_ids = (
    validation_queries_with_geo["search_location_id"]
    .fillna(-1).astype(np.int64).to_numpy()
)
validation_category_ids = (
    validation_queries_with_geo["search_category"]
    .fillna(-1).astype(np.int64).to_numpy()
)

np.save(ARTIFACTS_DIR / "candidate_item_ids.npy", candidate_item_ids)
np.save(ARTIFACTS_DIR / "candidate_location_ids.npy", candidate_location_ids)
np.save(ARTIFACTS_DIR / "candidate_category_ids.npy", candidate_category_ids)
np.save(ARTIFACTS_DIR / "validation_location_ids.npy", validation_location_ids)
np.save(ARTIFACTS_DIR / "validation_category_ids.npy", validation_category_ids)


In [ ]:
global_indices = np.load(
    ARTIFACTS_DIR / "local_semantic_top100_indices.npy",
    mmap_mode="r",
)
candidate_item_ids = np.load(
    ARTIFACTS_DIR / "candidate_item_ids.npy",
    mmap_mode="r",
)

assert np.array_equal(
    np.asarray(candidate_item_ids),
    candidate_items["item_id"].astype(str).to_numpy(),
)
assert global_indices.shape[0] == len(validation_queries_with_geo)
assert global_indices.min() >= 0
assert global_indices.max() < len(candidate_item_ids)


In [ ]:
def calculate_recall_scores(index_matrix, k=50):
    scores = np.empty(
        len(validation_queries_with_geo),
        dtype=np.float32,
    )

    for query_position, relevant_items in enumerate(
        validation_queries_with_geo["relevant_item_ids"]
    ):
        positions = np.asarray(
            index_matrix[query_position, :k]
        )

        positions = positions[positions >= 0]

        predicted_items = set(
            candidate_item_ids[positions]
        )

        relevant_items = set(relevant_items)

        scores[query_position] = (
            len(predicted_items & relevant_items)
            / len(relevant_items)
        )

    return scores

In [ ]:
global_recall_scores = calculate_recall_scores(global_indices, k=50)
global_recall_scores.mean(), (global_recall_scores > 0).mean()


(np.float32(0.14821285), np.float64(0.16345760512321386))

In [ ]:
global_recall = global_recall_scores.mean()
global_hit_rate = (global_recall_scores > 0).mean()

global_recall, global_hit_rate


(np.float32(0.14821285), np.float64(0.16345760512321386))

In [ ]:
seen_mask = validation_queries_with_geo["query_text_seen"].to_numpy()

pd.DataFrame({
    "query_group": [
        "Встречался в development",
        "Не встречался в development",
    ],
    "queries": [
        seen_mask.sum(),
        (~seen_mask).sum(),
    ],
    "recall_at_50": [
        global_recall_scores[seen_mask].mean(),
        global_recall_scores[~seen_mask].mean(),
    ],
    "hit_rate_at_50": [
        (global_recall_scores[seen_mask] > 0).mean(),
        (global_recall_scores[~seen_mask] > 0).mean(),
    ],
})


,query_group,queries,recall_at_50,hit_rate_at_50
0,Встречался в development,60532,0.135241,0.152184
1,Не встречался в development,10361,0.224000,0.229321


In [ ]:
relevant_pair_statistics = pd.Series({
    "same_location": (
        validation_rows["search_location_id"]
        == validation_rows["item_location_id"]
    ).mean(),

    "same_category": (
        validation_rows["search_category"]
        == validation_rows["item_category_id"]
    ).mean(),

    "same_location_and_category": (
        (
            validation_rows["search_location_id"]
            == validation_rows["item_location_id"]
        )
        & (
            validation_rows["search_category"]
            == validation_rows["item_category_id"]
        )
    ).mean(),
})

relevant_pair_statistics

same_location                 0.828279
same_category                 0.999939
same_location_and_category    0.828238
dtype: float64

Глобальный E5 слаб для общих услуг: выдачу занимают похожие объявления из других
городов. Простое поднятие локальных объектов внутри глобального top-100 повысило
Recall@50 только с 0.1482 до 0.1998; медиана локальных объектов в top-100 равнялась
нулю. Поэтому нужен отдельный локальный retrieval.


## 7. Точный локальный E5-поиск


Для каждой пары `(location_id, category_id)` строится небольшой `IndexFlatIP` и
извлекается top-40. Функция ниже предназначена для offline-запуска в чистом ядре:
так в памяти не находятся одновременно train, модель E5 и FAISS-индексы.


In [ ]:
def build_local_semantic_artifacts(artifacts_dir, top_k=40):
    """Build exact E5 neighbours inside each (location, category) group.

    Run this function in a clean kernel when the artifacts do not exist. It loads
    only memory-mapped arrays, so train and the E5 model do not occupy RAM.
    """
    item_embeddings = np.load(
        artifacts_dir / "local_item_embeddings_e5_small.npy", mmap_mode="r"
    )
    query_embeddings = np.load(
        artifacts_dir / "local_validation_query_embeddings_e5_small.npy",
        mmap_mode="r",
    )
    item_locations = np.load(
        artifacts_dir / "candidate_location_ids.npy", mmap_mode="r"
    )
    item_categories = np.load(
        artifacts_dir / "candidate_category_ids.npy", mmap_mode="r"
    )
    query_locations = np.load(
        artifacts_dir / "validation_location_ids.npy", mmap_mode="r"
    )
    query_categories = np.load(
        artifacts_dir / "validation_category_ids.npy", mmap_mode="r"
    )

    item_groups = pd.DataFrame(
        {"location": item_locations, "category": item_categories}
    ).groupby(["location", "category"], sort=False).indices
    query_groups = pd.DataFrame(
        {"location": query_locations, "category": query_categories}
    ).groupby(["location", "category"], sort=False).indices

    indices_path = artifacts_dir / "local_semantic_exact_top40_indices.npy"
    scores_path = artifacts_dir / "local_semantic_exact_top40_scores.npy"
    working_indices = artifacts_dir / "local_semantic_top40_indices.working.npy"
    working_scores = artifacts_dir / "local_semantic_top40_scores.working.npy"

    result_indices = np.lib.format.open_memmap(
        working_indices,
        mode="w+",
        dtype=np.int32,
        shape=(len(query_embeddings), top_k),
    )
    result_scores = np.lib.format.open_memmap(
        working_scores,
        mode="w+",
        dtype=np.float32,
        shape=(len(query_embeddings), top_k),
    )
    result_indices[:] = -1
    result_scores[:] = -np.inf
    faiss.omp_set_num_threads(4)

    for group_number, (group_key, query_positions) in enumerate(query_groups.items()):
        item_positions = item_groups.get(group_key)
        if item_positions is None:
            continue

        item_positions = np.asarray(item_positions, dtype=np.int64)
        query_positions = np.asarray(query_positions, dtype=np.int64)
        neighbours = min(top_k, len(item_positions))
        local_index = faiss.IndexFlatIP(item_embeddings.shape[1])

        for start in range(0, len(item_positions), 5_000):
            positions = item_positions[start:start + 5_000]
            local_index.add(
                np.ascontiguousarray(item_embeddings[positions], dtype=np.float32)
            )

        for start in range(0, len(query_positions), 500):
            positions = query_positions[start:start + 500]
            scores, relative_indices = local_index.search(
                np.ascontiguousarray(query_embeddings[positions], dtype=np.float32),
                neighbours,
            )
            result_indices[positions, :neighbours] = item_positions[relative_indices]
            result_scores[positions, :neighbours] = scores

        if group_number % 100 == 0:
            result_indices.flush()
            result_scores.flush()
            gc.collect()

    result_indices.flush()
    result_scores.flush()
    del result_indices, result_scores
    working_indices.replace(indices_path)
    working_scores.replace(scores_path)


In [ ]:
LOCAL_INDICES_PATH = ARTIFACTS_DIR / "local_semantic_exact_top40_indices.npy"
LOCAL_SCORES_PATH = ARTIFACTS_DIR / "local_semantic_exact_top40_scores.npy"

if not LOCAL_INDICES_PATH.exists() or not LOCAL_SCORES_PATH.exists():
    raise FileNotFoundError(
        "Перезапустите ядро и вызовите build_local_semantic_artifacts(ARTIFACTS_DIR)."
    )

local_indices = np.load(LOCAL_INDICES_PATH, mmap_mode="r")
local_scores = np.load(LOCAL_SCORES_PATH, mmap_mode="r")


In [ ]:
local_indices.shape, (local_indices[:, 0] >= 0).mean()

((70893, 40), np.float64(0.8735277107753939))

## 8. Гибрид 40 локальных + 10 глобальных


In [ ]:
candidate_category_ids = np.load(
    ARTIFACTS_DIR / "candidate_category_ids.npy", mmap_mode="r"
)
query_category_ids = np.load(
    ARTIFACTS_DIR / "validation_category_ids.npy", mmap_mode="r"
)

hybrid_indices = np.full(
    (len(validation_queries_with_geo), 50),
    -1,
    dtype=np.int32,
)

for query_position in range(len(validation_queries_with_geo)):
    selected = []
    selected_set = set()

    # Основной канал: до 40 объявлений той же категории и точной локации.
    for item_position in local_indices[query_position]:
        if item_position < 0:
            continue
        item_position = int(item_position)
        if item_position not in selected_set:
            selected.append(item_position)
            selected_set.add(item_position)

    # Fallback: глобальные E5-кандидаты правильной категории.
    for item_position in global_indices[query_position]:
        item_position = int(item_position)
        if (
            candidate_category_ids[item_position] == query_category_ids[query_position]
            and item_position not in selected_set
        ):
            selected.append(item_position)
            selected_set.add(item_position)
        if len(selected) == 50:
            break

    # Редкое несовпадение категории не должно оставлять пустые места.
    for item_position in global_indices[query_position]:
        if len(selected) == 50:
            break
        item_position = int(item_position)
        if item_position not in selected_set:
            selected.append(item_position)
            selected_set.add(item_position)

    hybrid_indices[query_position, :len(selected)] = selected


In [ ]:
hybrid_recall_scores = np.empty(
    len(validation_queries_with_geo),
    dtype=np.float32,
)

for query_position, relevant_items in enumerate(
    validation_queries_with_geo["relevant_item_ids"]
):
    positions = hybrid_indices[query_position]
    positions = positions[positions >= 0]

    predicted_item_ids = set(
        candidate_item_ids[positions]
    )

    relevant_item_ids = set(relevant_items)

    hybrid_recall_scores[query_position] = (
        len(predicted_item_ids & relevant_item_ids)
        / len(relevant_item_ids)
    )

hybrid_recall = hybrid_recall_scores.mean()
hybrid_hit_rate = (
    hybrid_recall_scores > 0
).mean()

hybrid_recall, hybrid_hit_rate

(np.float32(0.73322934), np.float64(0.7504831224521462))

In [ ]:
current_item_ids = (
    candidate_items["item_id"]
    .astype(str)
    .to_numpy()
)

np.array_equal(
    np.asarray(candidate_item_ids),
    current_item_ids,
)

True

In [ ]:
global_recall_scores = calculate_recall_scores(
    global_indices,
    k=50,
)

local_recall_scores = calculate_recall_scores(
    local_indices,
    k=40,
)

In [ ]:
pd.DataFrame({
    "method": [
        "Global E5",
        "Local E5 top-40",
        "Local 40 + global 10",
    ],
    "recall_at_50": [
        global_recall_scores.mean(),
        local_recall_scores.mean(),
        hybrid_recall_scores.mean(),
    ],
    "hit_rate_at_50": [
        (global_recall_scores > 0).mean(),
        (local_recall_scores > 0).mean(),
        (hybrid_recall_scores > 0).mean(),
    ],
})

,method,recall_at_50,hit_rate_at_50
0,Global E5,0.148213,0.163458
1,Local E5 top-40,0.700761,0.716121
2,Local 40 + global 10,0.733229,0.750483


### 8.1 Обобщение на новые тексты


In [ ]:
seen_mask = (
    validation_queries_with_geo["query_text_seen"]
    .to_numpy()
)

pd.DataFrame({
    "query_group": [
        "Встречался в development",
        "Не встречался в development",
    ],
    "queries": [
        seen_mask.sum(),
        (~seen_mask).sum(),
    ],
    "recall_at_50": [
        hybrid_recall_scores[seen_mask].mean(),
        hybrid_recall_scores[~seen_mask].mean(),
    ],
    "hit_rate_at_50": [
        (hybrid_recall_scores[seen_mask] > 0).mean(),
        (hybrid_recall_scores[~seen_mask] > 0).mean(),
    ],
})

,query_group,queries,recall_at_50,hit_rate_at_50
0,Встречался в development,60532,0.745541,0.764819
1,Не встречался в development,10361,0.661304,0.666731


На новых текстах Recall@50 равен 0.6613. Это подтверждает, что retrieval не сводится
к запоминанию популярных формулировок.


## 9. Анализ ошибок и географии


In [ ]:
item_location_by_id = (
    candidate_items
    .set_index("item_id")["item_location_id"]
)

has_local_relevant = []

for row in validation_queries_with_geo.itertuples():
    relevant_locations = (
        item_location_by_id
        .reindex(row.relevant_item_ids)
        .to_numpy()
    )

    has_local_relevant.append(
        np.any(
            relevant_locations
            == row.search_location_id
        )
    )

has_local_relevant = np.asarray(has_local_relevant)

In [ ]:
pd.DataFrame({
    "group": [
        "Есть релевантное в той же локации",
        "Все релевантные в других локациях",
    ],
    "queries": [
        has_local_relevant.sum(),
        (~has_local_relevant).sum(),
    ],
    "recall_at_50": [
        hybrid_recall_scores[
            has_local_relevant
        ].mean(),
        hybrid_recall_scores[
            ~has_local_relevant
        ].mean(),
    ],
    "hit_rate_at_50": [
        (
            hybrid_recall_scores[
                has_local_relevant
            ] > 0
        ).mean(),
        (
            hybrid_recall_scores[
                ~has_local_relevant
            ] > 0
        ).mean(),
    ],
})

,group,queries,recall_at_50,hit_rate_at_50
0,Есть релевантное в той же локации,57194,0.868783,0.887750
1,Все релевантные в других локациях,13699,0.167288,0.177385


Главная оставшаяся проблема — запросы, у которых все размеченные объявления находятся
в других локациях: их Recall@50 равен 0.1673 против 0.8688 для запросов с локальным
релевантным объектом.


### 9.1 Расстояние как дополнительный сигнал

Расстояние можно использовать только при наличии обеих координат. Оно не должно быть
жёстким фильтром: для значительной части межлокационных взаимодействий координаты
поисковой локации восстановить нельзя.


In [ ]:
# Для анализа расстояний используем уже заполненные query-координаты.
effective_query_centers = (
    validation_queries_with_geo[
        ["search_location_id", "query_latitude", "query_longitude"]
    ]
    .groupby("search_location_id", as_index=False)
    .agg(
        query_latitude=("query_latitude", "median"),
        query_longitude=("query_longitude", "median"),
    )
)

item_location_centers = (
    location_centers
    .reset_index()
    .rename(
        columns={
            "latitude": "item_location_latitude",
            "longitude": "item_location_longitude",
        }
    )
)

validation_pairs_with_geo = (
    validation_rows[
        [
            "search_location_id",
            "item_location_id",
            "item_latitude",
            "item_longitude",
        ]
    ]
    .merge(
        effective_query_centers,
        on="search_location_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        item_location_centers,
        on="item_location_id",
        how="left",
        validate="many_to_one",
    )
)

numeric_columns = [
    "query_latitude",
    "query_longitude",
    "item_latitude",
    "item_longitude",
    "item_location_latitude",
    "item_location_longitude",
]
validation_pairs_with_geo[numeric_columns] = (
    validation_pairs_with_geo[numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
)
validation_pairs_with_geo["effective_item_latitude"] = (
    validation_pairs_with_geo["item_latitude"]
    .fillna(validation_pairs_with_geo["item_location_latitude"])
)
validation_pairs_with_geo["effective_item_longitude"] = (
    validation_pairs_with_geo["item_longitude"]
    .fillna(validation_pairs_with_geo["item_location_longitude"])
)

cross_location_pairs = validation_pairs_with_geo[
    validation_pairs_with_geo["search_location_id"]
    != validation_pairs_with_geo["item_location_id"]
].dropna(
    subset=[
        "query_latitude",
        "query_longitude",
        "effective_item_latitude",
        "effective_item_longitude",
    ]
).copy()

cross_location_pairs["distance_km"] = haversine_distance(
    cross_location_pairs["query_latitude"].to_numpy(dtype=np.float64),
    cross_location_pairs["query_longitude"].to_numpy(dtype=np.float64),
    cross_location_pairs["effective_item_latitude"].to_numpy(dtype=np.float64),
    cross_location_pairs["effective_item_longitude"].to_numpy(dtype=np.float64),
)


In [ ]:
all_cross_pairs = (
    validation_rows["search_location_id"]
    != validation_rows["item_location_id"]
).sum()

radii = [10, 25, 50, 100, 300, 1_000]

pd.DataFrame({
    "radius_km": radii,
    "share_among_known_distances": [
        (cross_location_pairs["distance_km"] <= radius).mean()
        for radius in radii
    ],
}).assign(
    coordinate_coverage=len(cross_location_pairs) / all_cross_pairs
)


,radius_km,share_among_known_distances,coordinate_coverage
0,10,0.345560,0.998352
1,25,0.606263,0.998352
2,50,0.738884,0.998352
3,100,0.821736,0.998352
4,300,0.895035,0.998352
5,1000,0.951115,0.998352


После применения development-fallback расстояние удалось вычислить для **99.84%**
межлокационных пар. Среди них 82.2% находятся в пределах 100 км. Это подтверждает,
что точный `location_id` слишком строг: соседние локации стоит добавить отдельным
географическим каналом, сохранив глобальный fallback для дальних услуг.


## 10. Следующий эксперимент: TF-IDF + E5

Текущий checkpoint: **Recall@50 = 0.7332**. Следующий шаг — получить локальные и
глобальные TF-IDF-кандидаты, затем проверить квоты между лексическим и семантическим
каналами. TF-IDF должен дополнить E5 на редких названиях, артикулах, аббревиатурах и
точных формулировках параметров.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

tfidf_item_matrix = load_npz(
    ARTIFACTS_DIR / "tfidf_item_matrix.npz"
).tocsr()

tfidf_query_matrix = load_npz(
    ARTIFACTS_DIR / "tfidf_validation_query_matrix.npz"
).tocsr()

item_location_ids = np.load(
    ARTIFACTS_DIR / "candidate_location_ids.npy",
    mmap_mode="r",
)

item_category_ids = np.load(
    ARTIFACTS_DIR / "candidate_category_ids.npy",
    mmap_mode="r",
)

query_location_ids = np.load(
    ARTIFACTS_DIR / "validation_location_ids.npy",
    mmap_mode="r",
)

query_category_ids = np.load(
    ARTIFACTS_DIR / "validation_category_ids.npy",
    mmap_mode="r",
)

In [ ]:
(
    tfidf_item_matrix.shape,
    tfidf_query_matrix.shape,
    item_location_ids.shape,
    query_location_ids.shape,
)

((344825, 172789), (70893, 172789), (344825,), (70893,))

In [ ]:
item_groups = pd.DataFrame({
    "location": item_location_ids,
    "category": item_category_ids,
}).groupby(
    ["location", "category"],
    sort=False,
).indices

query_groups = pd.DataFrame({
    "location": query_location_ids,
    "category": query_category_ids,
}).groupby(
    ["location", "category"],
    sort=False,
).indices

len(item_groups), len(query_groups)

(2647, 1814)

In [ ]:
TFIDF_LOCAL_INDICES_PATH = (
    ARTIFACTS_DIR / "local_tfidf_top40_indices.npy"
)
TFIDF_LOCAL_SCORES_PATH = (
    ARTIFACTS_DIR / "local_tfidf_top40_scores.npy"
)

top_k = 40

local_tfidf_indices = np.lib.format.open_memmap(
    TFIDF_LOCAL_INDICES_PATH,
    mode="w+",
    dtype=np.int32,
    shape=(tfidf_query_matrix.shape[0], top_k),
)
local_tfidf_indices[:] = -1

local_tfidf_scores = np.lib.format.open_memmap(
    TFIDF_LOCAL_SCORES_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(tfidf_query_matrix.shape[0], top_k),
)
local_tfidf_scores[:] = -np.inf

In [ ]:
for group_number, (group_key, query_positions) in enumerate(
    query_groups.items()
):
    item_positions = item_groups.get(group_key)

    if item_positions is None:
        continue

    item_positions = np.asarray(item_positions)
    query_positions = np.asarray(query_positions)

    neighbors_count = min(top_k, len(item_positions))

    search_index = NearestNeighbors(
        metric="cosine",
        algorithm="brute",
        n_jobs=4,
    )

    search_index.fit(
        tfidf_item_matrix[item_positions]
    )

    for start in range(0, len(query_positions), 500):
        batch_positions = query_positions[start:start + 500]

        distances, relative_indices = (
            search_index.kneighbors(
                tfidf_query_matrix[batch_positions],
                n_neighbors=neighbors_count,
            )
        )

        local_tfidf_indices[
            batch_positions,
            :neighbors_count,
        ] = item_positions[relative_indices]

        local_tfidf_scores[
            batch_positions,
            :neighbors_count,
        ] = 1 - distances

    if group_number % 100 == 0:
        local_tfidf_indices.flush()
        local_tfidf_scores.flush()

        print(
            f"Обработано {group_number} "
            f"из {len(query_groups)} групп"
        )

Обработано 0 из 1814 групп
Обработано 100 из 1814 групп
Обработано 200 из 1814 групп
Обработано 300 из 1814 групп
Обработано 400 из 1814 групп
Обработано 600 из 1814 групп
Обработано 700 из 1814 групп
Обработано 800 из 1814 групп
Обработано 900 из 1814 групп
Обработано 1000 из 1814 групп
Обработано 1100 из 1814 групп
Обработано 1200 из 1814 групп
Обработано 1300 из 1814 групп
Обработано 1400 из 1814 групп
Обработано 1500 из 1814 групп
Обработано 1600 из 1814 групп
Обработано 1700 из 1814 групп
Обработано 1800 из 1814 групп


In [ ]:
local_tfidf_indices.flush()
local_tfidf_scores.flush()

local_tfidf_indices.shape, (
    local_tfidf_indices[:, 0] >= 0
).mean()

((70893, 40), np.float64(0.8735277107753939))

In [ ]:
tfidf_local_indices = np.load(
    ARTIFACTS_DIR / "local_tfidf_top40_indices.npy",
    mmap_mode="r",
)

e5_local_indices = np.load(
    ARTIFACTS_DIR / "local_semantic_exact_top40_indices.npy",
    mmap_mode="r",
)

e5_global_indices = np.load(
    ARTIFACTS_DIR / "local_semantic_top100_indices.npy",
    mmap_mode="r",
)

candidate_item_ids = np.load(
    ARTIFACTS_DIR / "candidate_item_ids.npy",
    mmap_mode="r",
)

In [ ]:
assert tfidf_local_indices.shape == (70893, 40)
assert e5_local_indices.shape == (70893, 40)
assert e5_global_indices.shape == (70893, 100)

assert np.array_equal(
    np.asarray(candidate_item_ids),
    candidate_items["item_id"].astype(str).to_numpy(),
)

In [ ]:
def calculate_recall_scores(
    index_matrix,
    relevant_items_per_query,
    k=50,
):
    scores = np.empty(
        len(relevant_items_per_query),
        dtype=np.float32,
    )

    for query_position, relevant_items in enumerate(
        relevant_items_per_query
    ):
        positions = np.asarray(
            index_matrix[query_position, :k]
        )

        positions = positions[positions >= 0]

        predicted_items = set(
            candidate_item_ids[positions]
        )

        relevant_items = set(relevant_items)

        scores[query_position] = (
            len(predicted_items & relevant_items)
            / len(relevant_items)
        )

    return scores

In [ ]:
relevant_items_per_query = (
    validation_queries["relevant_item_ids"]
    .tolist()
)

In [ ]:
tfidf_local_recall_scores = calculate_recall_scores(
    tfidf_local_indices,
    relevant_items_per_query,
    k=40,
)

e5_local_recall_scores = calculate_recall_scores(
    e5_local_indices,
    relevant_items_per_query,
    k=40,
)

In [ ]:
pd.DataFrame({
    "method": [
        "Local TF-IDF top-40",
        "Local E5 top-40",
    ],
    "recall_at_40": [
        tfidf_local_recall_scores.mean(),
        e5_local_recall_scores.mean(),
    ],
    "hit_rate_at_40": [
        (tfidf_local_recall_scores > 0).mean(),
        (e5_local_recall_scores > 0).mean(),
    ],
})

,method,recall_at_40,hit_rate_at_40
0,Local TF-IDF top-40,0.580258,0.602556
1,Local E5 top-40,0.700761,0.716121


In [ ]:
e5_hits = e5_local_recall_scores > 0
tfidf_hits = tfidf_local_recall_scores > 0

pd.Series({
    "нашли оба": (e5_hits & tfidf_hits).sum(),
    "нашёл только E5": (e5_hits & ~tfidf_hits).sum(),
    "нашёл только TF-IDF": (~e5_hits & tfidf_hits).sum(),
    "не нашёл никто": (~e5_hits & ~tfidf_hits).sum(),
})

нашли оба              40844
нашёл только E5         9924
нашёл только TF-IDF     1873
не нашёл никто         18252
dtype: int64

In [ ]:
candidate_category_ids = np.load(
    ARTIFACTS_DIR / "candidate_category_ids.npy",
    mmap_mode="r",
)

query_category_ids = np.load(
    ARTIFACTS_DIR / "validation_category_ids.npy",
    mmap_mode="r",
)

In [ ]:
def build_fused_candidates(
    e5_local_quota,
    tfidf_local_quota,
    total_candidates=50,
):
    result = np.full(
        (len(validation_queries), total_candidates),
        -1,
        dtype=np.int32,
    )

    for query_position in range(len(validation_queries)):
        selected = []
        selected_set = set()

        def add_unique(source, quota):
            if quota <= 0:
                return

            added = 0

            for item_position in source:
                if item_position < 0:
                    continue

                item_position = int(item_position)

                if item_position not in selected_set:
                    selected.append(item_position)
                    selected_set.add(item_position)
                    added += 1

                if (
                    added >= quota
                    or len(selected) >= total_candidates
                ):
                    break

        add_unique(
            e5_local_indices[query_position],
            e5_local_quota,
        )

        add_unique(
            tfidf_local_indices[query_position],
            tfidf_local_quota,
        )

        # Глобальные E5-кандидаты правильной категории.
        for item_position in e5_global_indices[query_position]:
            if len(selected) >= total_candidates:
                break

            item_position = int(item_position)

            if (
                candidate_category_ids[item_position]
                == query_category_ids[query_position]
                and item_position not in selected_set
            ):
                selected.append(item_position)
                selected_set.add(item_position)

        # Заполняем остаток без фильтра категории.
        for item_position in e5_global_indices[query_position]:
            if len(selected) >= total_candidates:
                break

            item_position = int(item_position)

            if item_position not in selected_set:
                selected.append(item_position)
                selected_set.add(item_position)

        result[
            query_position,
            :len(selected),
        ] = selected

    return result

In [ ]:
quota_options = [
    (40, 0),
    (35, 5),
    (30, 10),
    (25, 15),
    (20, 20),
]

fusion_results = []

for e5_quota, tfidf_quota in quota_options:
    fused_indices = build_fused_candidates(
        e5_local_quota=e5_quota,
        tfidf_local_quota=tfidf_quota,
    )

    recall_scores = calculate_recall_scores(
        fused_indices,
        relevant_items_per_query,
        k=50,
    )

    fusion_results.append({
        "e5_local": e5_quota,
        "tfidf_local": tfidf_quota,
        "global_fallback": 10,
        "recall_at_50": recall_scores.mean(),
        "hit_rate_at_50": (recall_scores > 0).mean(),
    })

pd.DataFrame(fusion_results)

,e5_local,tfidf_local,global_fallback,recall_at_50,hit_rate_at_50
0,40,0,10,0.733229,0.750483
1,35,5,10,0.738701,0.755589
2,30,10,10,0.736955,0.754108
3,25,15,10,0.732911,0.750540
4,20,20,10,0.726943,0.745419


In [ ]:
best_fused_indices = build_fused_candidates(
    e5_local_quota=35,
    tfidf_local_quota=5,
)

best_fused_recall_scores = calculate_recall_scores(
    best_fused_indices,
    relevant_items_per_query,
    k=50,
)

np.save(
    ARTIFACTS_DIR / "best_e5_tfidf_indices.npy",
    best_fused_indices,
)

best_fused_recall_scores.mean(), (
    best_fused_recall_scores > 0
).mean()

(np.float32(0.7387008), np.float64(0.7555894093916183))

In [ ]:
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

item_latitudes = np.load(
    ARTIFACTS_DIR / "candidate_latitudes.npy",
    mmap_mode="r",
)
item_longitudes = np.load(
    ARTIFACTS_DIR / "candidate_longitudes.npy",
    mmap_mode="r",
)
query_latitudes = np.load(
    ARTIFACTS_DIR / "validation_query_latitudes.npy",
    mmap_mode="r",
)
query_longitudes = np.load(
    ARTIFACTS_DIR / "validation_query_longitudes.npy",
    mmap_mode="r",
)

item_location_ids = np.load(
    ARTIFACTS_DIR / "candidate_location_ids.npy",
    mmap_mode="r",
)
item_category_ids = np.load(
    ARTIFACTS_DIR / "candidate_category_ids.npy",
    mmap_mode="r",
)
query_location_ids = np.load(
    ARTIFACTS_DIR / "validation_location_ids.npy",
    mmap_mode="r",
)
query_category_ids = np.load(
    ARTIFACTS_DIR / "validation_category_ids.npy",
    mmap_mode="r",
)

In [ ]:
valid_item_mask = (
    np.isfinite(item_latitudes)
    & np.isfinite(item_longitudes)
)

valid_item_positions = np.flatnonzero(
    valid_item_mask
)

item_coordinates_radians = np.radians(
    np.column_stack([
        item_latitudes[valid_item_positions],
        item_longitudes[valid_item_positions],
    ])
)

geo_index = BallTree(
    item_coordinates_radians,
    metric="haversine",
    leaf_size=40,
)

In [ ]:
query_groups = pd.DataFrame({
    "location": query_location_ids,
    "category": query_category_ids,
}).groupby(
    ["location", "category"],
    sort=False,
).indices

In [ ]:
(
    len(valid_item_positions),
    len(item_latitudes),
    len(query_groups),
)

(344825, 344825, 1814)

In [ ]:
import gc

item_embeddings = np.load(
    ARTIFACTS_DIR / "local_item_embeddings_e5_small.npy",
    mmap_mode="r",
)

query_embeddings = np.load(
    ARTIFACTS_DIR / "local_validation_query_embeddings_e5_small.npy",
    mmap_mode="r",
)

NEARBY_INDICES_PATH = (
    ARTIFACTS_DIR / "nearby_semantic_top20_indices.npy"
)
WORKING_PATH = (
    ARTIFACTS_DIR / "nearby_semantic_top20_indices.working.npy"
)

nearby_top_k = 20
radius_radians = 100 / 6371.0

In [ ]:
if NEARBY_INDICES_PATH.exists():
    nearby_indices = np.load(
        NEARBY_INDICES_PATH,
        mmap_mode="r",
    )
else:
    nearby_indices = np.lib.format.open_memmap(
        WORKING_PATH,
        mode="w+",
        dtype=np.int32,
        shape=(len(query_embeddings), nearby_top_k),
    )
    nearby_indices[:] = -1

    faiss.omp_set_num_threads(4)

    for group_number, (
        group_key,
        query_positions,
    ) in enumerate(query_groups.items()):
        location_id, category_id = group_key

        query_positions = np.asarray(
            query_positions,
            dtype=np.int64,
        )

        finite_coordinates = (
            np.isfinite(query_latitudes[query_positions])
            & np.isfinite(query_longitudes[query_positions])
        )

        if not finite_coordinates.any():
            continue

        coordinate_positions = query_positions[
            finite_coordinates
        ]

        query_center = np.radians([[
            np.nanmedian(
                query_latitudes[coordinate_positions]
            ),
            np.nanmedian(
                query_longitudes[coordinate_positions]
            ),
        ]])

        relative_positions = geo_index.query_radius(
            query_center,
            r=radius_radians,
        )[0]

        candidate_positions = valid_item_positions[
            relative_positions
        ]

        candidate_positions = candidate_positions[
            (
                item_category_ids[candidate_positions]
                == category_id
            )
            & (
                item_location_ids[candidate_positions]
                != location_id
            )
        ]

        if len(candidate_positions) == 0:
            continue

        neighbors_count = min(
            nearby_top_k,
            len(candidate_positions),
        )

        semantic_index = faiss.IndexFlatIP(
            item_embeddings.shape[1]
        )

        for start in range(
            0,
            len(candidate_positions),
            5_000,
        ):
            positions = candidate_positions[
                start:start + 5_000
            ]

            semantic_index.add(
                np.ascontiguousarray(
                    item_embeddings[positions],
                    dtype=np.float32,
                )
            )

        for start in range(0, len(query_positions), 500):
            positions = query_positions[start:start + 500]

            _, relative_indices = semantic_index.search(
                np.ascontiguousarray(
                    query_embeddings[positions],
                    dtype=np.float32,
                ),
                neighbors_count,
            )

            nearby_indices[
                positions,
                :neighbors_count,
            ] = candidate_positions[relative_indices]

        del semantic_index

        if group_number % 100 == 0:
            nearby_indices.flush()
            gc.collect()

            print(
                f"Обработано {group_number} "
                f"из {len(query_groups)} групп"
            )

    nearby_indices.flush()
    del nearby_indices

    WORKING_PATH.replace(NEARBY_INDICES_PATH)

    nearby_indices = np.load(
        NEARBY_INDICES_PATH,
        mmap_mode="r",
    )

Обработано 0 из 1814 групп
Обработано 100 из 1814 групп
Обработано 200 из 1814 групп
Обработано 300 из 1814 групп
Обработано 400 из 1814 групп
Обработано 500 из 1814 групп
Обработано 600 из 1814 групп
Обработано 700 из 1814 групп
Обработано 800 из 1814 групп
Обработано 900 из 1814 групп
Обработано 1000 из 1814 групп
Обработано 1100 из 1814 групп
Обработано 1200 из 1814 групп
Обработано 1300 из 1814 групп
Обработано 1400 из 1814 групп
Обработано 1500 из 1814 групп
Обработано 1600 из 1814 групп
Обработано 1700 из 1814 групп
Обработано 1800 из 1814 групп


In [ ]:
nearby_counts = (nearby_indices >= 0).sum(axis=1)

nearby_indices.shape, pd.Series(nearby_counts).describe()

((70893, 20),
 count    70893.000000
 mean        19.623291
 std          2.371887
 min          0.000000
 25%         20.000000
 50%         20.000000
 75%         20.000000
 max         20.000000
 dtype: float64)

In [ ]:
local_e5 = np.load(
    ARTIFACTS_DIR / "local_semantic_exact_top40_indices.npy",
    mmap_mode="r",
)
local_tfidf = np.load(
    ARTIFACTS_DIR / "local_tfidf_top40_indices.npy",
    mmap_mode="r",
)
nearby_e5 = np.load(
    ARTIFACTS_DIR / "nearby_semantic_top20_indices.npy",
    mmap_mode="r",
)
global_e5 = np.load(
    ARTIFACTS_DIR / "local_semantic_top100_indices.npy",
    mmap_mode="r",
)

candidate_item_ids = np.load(
    ARTIFACTS_DIR / "candidate_item_ids.npy",
    mmap_mode="r",
)
candidate_category_ids = np.load(
    ARTIFACTS_DIR / "candidate_category_ids.npy",
    mmap_mode="r",
)
query_category_ids = np.load(
    ARTIFACTS_DIR / "validation_category_ids.npy",
    mmap_mode="r",
)

In [ ]:
def build_geo_fusion(
    local_e5_quota,
    local_tfidf_quota,
    nearby_e5_quota,
):
    result = np.full(
        (len(validation_queries), 50),
        -1,
        dtype=np.int32,
    )

    for query_position in range(len(validation_queries)):
        selected = []
        selected_set = set()

        def add_unique(source, quota):
            added = 0

            for position in source:
                if position < 0 or added >= quota:
                    break

                position = int(position)

                if position not in selected_set:
                    selected.append(position)
                    selected_set.add(position)
                    added += 1

        add_unique(
            local_e5[query_position],
            local_e5_quota,
        )
        add_unique(
            local_tfidf[query_position],
            local_tfidf_quota,
        )
        add_unique(
            nearby_e5[query_position],
            nearby_e5_quota,
        )

        # Глобальный E5 заполняет все оставшиеся места.
        for position in global_e5[query_position]:
            if len(selected) >= 50:
                break

            position = int(position)

            if (
                candidate_category_ids[position]
                == query_category_ids[query_position]
                and position not in selected_set
            ):
                selected.append(position)
                selected_set.add(position)

        for position in global_e5[query_position]:
            if len(selected) >= 50:
                break

            position = int(position)

            if position not in selected_set:
                selected.append(position)
                selected_set.add(position)

        result[query_position, :len(selected)] = selected

    return result

In [ ]:
def evaluate_indices(indices):
    scores = np.empty(
        len(validation_queries),
        dtype=np.float32,
    )

    for query_position, relevant_items in enumerate(
        validation_queries["relevant_item_ids"]
    ):
        positions = indices[query_position]
        positions = positions[positions >= 0]

        predicted = set(candidate_item_ids[positions])
        relevant = set(relevant_items)

        scores[query_position] = (
            len(predicted & relevant) / len(relevant)
        )

    return scores

In [ ]:
quota_options = [
    (35, 5, 0),   # предыдущий baseline
    (35, 5, 5),
    (30, 5, 5),
    (30, 5, 10),
    (25, 5, 10),
    (25, 5, 15),
    (30, 0, 10),
]

geo_fusion_results = []

for e5_quota, tfidf_quota, nearby_quota in quota_options:
    indices = build_geo_fusion(
        e5_quota,
        tfidf_quota,
        nearby_quota,
    )
    scores = evaluate_indices(indices)

    geo_fusion_results.append({
        "local_e5": e5_quota,
        "local_tfidf": tfidf_quota,
        "nearby_e5": nearby_quota,
        "nominal_global": (
            50 - e5_quota - tfidf_quota - nearby_quota
        ),
        "recall_at_50": scores.mean(),
        "hit_rate_at_50": (scores > 0).mean(),
    })

pd.DataFrame(geo_fusion_results)

,local_e5,local_tfidf,nearby_e5,nominal_global,recall_at_50,hit_rate_at_50
0,35,5,0,10,0.738701,0.755589
1,35,5,5,5,0.776748,0.793746
2,30,5,5,10,0.768255,0.786368
3,30,5,10,5,0.784618,0.802632
4,25,5,10,10,0.773706,0.793083
5,25,5,15,5,0.783775,0.803098
6,30,0,10,10,0.768873,0.788470


In [ ]:
best_geo_indices = build_geo_fusion(
    local_e5_quota=30,
    local_tfidf_quota=5,
    nearby_e5_quota=10,
)

best_geo_recall_scores = evaluate_indices(
    best_geo_indices
)

np.save(
    ARTIFACTS_DIR / "best_geo_e5_tfidf_indices.npy",
    best_geo_indices,
)

best_geo_recall_scores.mean(), (
    best_geo_recall_scores > 0
).mean()

(np.float32(0.78461766), np.float64(0.8026321357538826))

In [ ]:
item_location_by_id = (
    candidate_items
    .set_index("item_id")["item_location_id"]
)

has_local_relevant = []

for row in validation_queries.itertuples():
    relevant_locations = (
        item_location_by_id
        .reindex(row.relevant_item_ids)
        .to_numpy()
    )

    has_local_relevant.append(
        np.any(
            relevant_locations
            == row.search_location_id
        )
    )

has_local_relevant = np.asarray(
    has_local_relevant
)

In [ ]:
pd.DataFrame({
    "group": [
        "Есть релевантное в точной локации",
        "Все релевантные в других локациях",
    ],
    "queries": [
        has_local_relevant.sum(),
        (~has_local_relevant).sum(),
    ],
    "recall_at_50": [
        best_geo_recall_scores[
            has_local_relevant
        ].mean(),
        best_geo_recall_scores[
            ~has_local_relevant
        ].mean(),
    ],
    "hit_rate_at_50": [
        (
            best_geo_recall_scores[
                has_local_relevant
            ] > 0
        ).mean(),
        (
            best_geo_recall_scores[
                ~has_local_relevant
            ] > 0
        ).mean(),
    ],
})

,group,queries,recall_at_50,hit_rate_at_50
0,Есть релевантное в точной локации,57194,0.865696,0.884254
1,Все релевантные в других локациях,13699,0.446112,0.461859


In [ ]:
channel_matrices = [
    local_e5,
    local_tfidf,
    nearby_e5,
    global_e5,
]

pool_recall_scores = np.empty(
    len(validation_queries),
    dtype=np.float32,
)

oracle_recall_at_50_scores = np.empty(
    len(validation_queries),
    dtype=np.float32,
)

pool_sizes = np.empty(
    len(validation_queries),
    dtype=np.int32,
)

In [ ]:
for query_position, relevant_items in enumerate(
    validation_queries["relevant_item_ids"]
):
    union_positions = set()

    for channel_matrix in channel_matrices:
        for item_position in channel_matrix[
            query_position
        ]:
            if item_position >= 0:
                union_positions.add(
                    int(item_position)
                )

    pool_sizes[query_position] = len(
        union_positions
    )

    predicted_item_ids = set(
        candidate_item_ids[
            list(union_positions)
        ]
    )

    relevant_item_ids = set(
        relevant_items
    )

    found_relevant = len(
        predicted_item_ids & relevant_item_ids
    )

    # Recall всего объединённого пула.
    pool_recall_scores[query_position] = (
        found_relevant
        / len(relevant_item_ids)
    )

    # Идеальный агрегатор может поместить в top-50
    # не больше 50 релевантных объектов.
    oracle_recall_at_50_scores[query_position] = (
        min(found_relevant, 50)
        / len(relevant_item_ids)
    )

In [ ]:
pd.Series({
    "mean_pool_size": pool_sizes.mean(),
    "median_pool_size": np.median(pool_sizes),
    "max_pool_size": pool_sizes.max(),

    "union_recall": pool_recall_scores.mean(),
    "oracle_recall_at_50": (
        oracle_recall_at_50_scores.mean()
    ),

    "union_hit_rate": (
        pool_recall_scores > 0
    ).mean(),
})

mean_pool_size         170.453895
median_pool_size       179.000000
max_pool_size          200.000000
union_recall             0.842042
oracle_recall_at_50      0.842042
union_hit_rate           0.856530
dtype: float64

In [ ]:
from sklearn.model_selection import train_test_split

development_query_texts = set(
    development_train["search_query"]
)

validation_queries["query_text_seen"] = (
    validation_queries["search_query"]
    .isin(development_query_texts)
)

validation_queries[
    "query_text_seen"
].value_counts()
all_query_positions = np.arange(
    len(validation_queries)
)

tuning_positions, test_positions = train_test_split(
    all_query_positions,
    test_size=0.5,
    random_state=42,
    stratify=validation_queries["query_text_seen"],
)

len(tuning_positions), len(test_positions)

(35446, 35447)

In [ ]:
rrf_channels = [
    local_e5,
    local_tfidf,
    nearby_e5,
    global_e5,
]

def build_weighted_rrf(
    weights,
    rrf_k=30,
):
    result = np.full(
        (len(validation_queries), 50),
        -1,
        dtype=np.int32,
    )

    for query_position in range(len(validation_queries)):
        scores = {}

        for channel, weight in zip(
            rrf_channels,
            weights,
        ):
            for rank, item_position in enumerate(
                channel[query_position],
                start=1,
            ):
                if item_position < 0:
                    continue

                item_position = int(item_position)

                scores[item_position] = (
                    scores.get(item_position, 0.0)
                    + weight / (rrf_k + rank)
                )

        ranked_positions = sorted(
            scores,
            key=scores.get,
            reverse=True,
        )

        # Почти все релевантные пары совпадают по категории.
        same_category = [
            position
            for position in ranked_positions
            if candidate_category_ids[position]
            == query_category_ids[query_position]
        ]

        other_categories = [
            position
            for position in ranked_positions
            if candidate_category_ids[position]
            != query_category_ids[query_position]
        ]

        selected = (
            same_category + other_categories
        )[:50]

        result[
            query_position,
            :len(selected),
        ] = selected

    return result

In [ ]:
rrf_configurations = [
    {
        "name": "equal_k60",
        "weights": (1.0, 1.0, 1.0, 1.0),
        "k": 60,
    },
    {
        "name": "equal_k20",
        "weights": (1.0, 1.0, 1.0, 1.0),
        "k": 20,
    },
    {
        "name": "local_priority",
        "weights": (2.0, 1.0, 1.5, 0.5),
        "k": 30,
    },
    {
        "name": "e5_priority",
        "weights": (3.0, 0.5, 1.5, 0.5),
        "k": 30,
    },
    {
        "name": "nearby_priority",
        "weights": (2.0, 1.0, 3.0, 0.5),
        "k": 30,
    },
    {
        "name": "balanced",
        "weights": (3.0, 1.0, 2.0, 0.5),
        "k": 30,
    },
]

In [ ]:
rrf_tuning_results = []

for configuration in rrf_configurations:
    rrf_indices = build_weighted_rrf(
        weights=configuration["weights"],
        rrf_k=configuration["k"],
    )

    recall_scores = evaluate_indices(
        rrf_indices
    )

    rrf_tuning_results.append({
        **configuration,
        "tuning_recall": recall_scores[
            tuning_positions
        ].mean(),
        "tuning_hit_rate": (
            recall_scores[tuning_positions] > 0
        ).mean(),
    })

rrf_tuning_table = pd.DataFrame(
    rrf_tuning_results
).sort_values(
    "tuning_recall",
    ascending=False,
)

rrf_tuning_table

,name,weights,k,tuning_recall,tuning_hit_rate
3,e5_priority,"(3.0, 0.5, 1.5, 0.5)",30,0.797582,0.815212
5,balanced,"(3.0, 1.0, 2.0, 0.5)",30,0.795925,0.813829
2,local_priority,"(2.0, 1.0, 1.5, 0.5)",30,0.794223,0.812560
4,nearby_priority,"(2.0, 1.0, 3.0, 0.5)",30,0.786690,0.805987
0,equal_k60,"(1.0, 1.0, 1.0, 1.0)",60,0.772959,0.794081
1,equal_k20,"(1.0, 1.0, 1.0, 1.0)",20,0.772837,0.793968


In [ ]:
best_rrf_config = rrf_tuning_table.iloc[0]

best_rrf_indices = build_weighted_rrf(
    weights=best_rrf_config["weights"],
    rrf_k=int(best_rrf_config["k"]),
)

best_rrf_recall_scores = evaluate_indices(
    best_rrf_indices
)

pd.Series({
    "quota_baseline_test_recall": (
        best_geo_recall_scores[
            test_positions
        ].mean()
    ),
    "rrf_test_recall": (
        best_rrf_recall_scores[
            test_positions
        ].mean()
    ),
    "rrf_test_hit_rate": (
        best_rrf_recall_scores[
            test_positions
        ] > 0
    ).mean(),
})

quota_baseline_test_recall    0.785920
rrf_test_recall               0.801750
rrf_test_hit_rate             0.818743
dtype: float64

In [ ]:
local_e5_scores = np.load(
    ARTIFACTS_DIR / "local_semantic_exact_top40_scores.npy",
    mmap_mode="r",
)

local_tfidf_scores = np.load(
    ARTIFACTS_DIR / "local_tfidf_top40_scores.npy",
    mmap_mode="r",
)

global_e5_scores = np.load(
    ARTIFACTS_DIR / "local_semantic_top100_scores.npy",
    mmap_mode="r",
)

In [ ]:
FEATURE_NAMES = [
    "local_e5_reciprocal_rank",
    "local_tfidf_reciprocal_rank",
    "nearby_e5_reciprocal_rank",
    "global_e5_reciprocal_rank",
    "in_local_e5",
    "in_local_tfidf",
    "in_nearby_e5",
    "in_global_e5",
    "local_e5_score",
    "local_tfidf_score",
    "global_e5_score",
    "channels_count",
    "same_location",
    "same_category",
]

In [ ]:
candidate_location_ids = np.load(
    ARTIFACTS_DIR / "candidate_location_ids.npy",
    mmap_mode="r",
)

query_location_ids = np.load(
    ARTIFACTS_DIR / "validation_location_ids.npy",
    mmap_mode="r",
)

candidate_category_ids = np.load(
    ARTIFACTS_DIR / "candidate_category_ids.npy",
    mmap_mode="r",
)

query_category_ids = np.load(
    ARTIFACTS_DIR / "validation_category_ids.npy",
    mmap_mode="r",
)

In [ ]:
def build_query_candidate_features(query_position):
    features_by_item = {}

    channel_specs = [
        (local_e5, local_e5_scores, 8),
        (local_tfidf, local_tfidf_scores, 9),
        (nearby_e5, None, None),
        (global_e5, global_e5_scores, 10),
    ]

    for channel_number, (
        indices,
        scores,
        score_column,
    ) in enumerate(channel_specs):
        for rank, item_position in enumerate(
            indices[query_position],
            start=1,
        ):
            if item_position < 0:
                continue

            item_position = int(item_position)

            if item_position not in features_by_item:
                features_by_item[item_position] = np.zeros(
                    len(FEATURE_NAMES),
                    dtype=np.float32,
                )

            features = features_by_item[item_position]

            features[channel_number] = 1.0 / rank
            features[4 + channel_number] = 1.0

            if score_column is not None:
                score = scores[
                    query_position,
                    rank - 1,
                ]

                if np.isfinite(score):
                    features[score_column] = score

    item_positions = np.fromiter(
        features_by_item.keys(),
        dtype=np.int32,
    )

    feature_matrix = np.vstack(
        list(features_by_item.values())
    )

    for row_number, item_position in enumerate(
        item_positions
    ):
        feature_matrix[row_number, 11] = (
            feature_matrix[row_number, 4:8].sum()
        )

        feature_matrix[row_number, 12] = (
            candidate_location_ids[item_position]
            == query_location_ids[query_position]
        )

        feature_matrix[row_number, 13] = (
            candidate_category_ids[item_position]
            == query_category_ids[query_position]
        )

    return item_positions, feature_matrix

In [ ]:
example_positions, example_features = (
    build_query_candidate_features(0)
)

example_positions.shape, example_features.shape

((179,), (179, 14))

In [ ]:
rng = np.random.default_rng(42)

train_feature_parts = []
train_label_parts = []

queries_with_positive = 0

for query_position in tuning_positions:
    item_positions, features = (
        build_query_candidate_features(
            query_position
        )
    )

    relevant_items = set(
        validation_queries.iloc[
            query_position
        ]["relevant_item_ids"]
    )

    labels = np.fromiter(
        (
            candidate_item_ids[item_position]
            in relevant_items
            for item_position in item_positions
        ),
        dtype=np.int8,
    )

    positive_rows = np.flatnonzero(labels == 1)
    negative_rows = np.flatnonzero(labels == 0)

    # Если положительного объекта нет в пуле,
    # агрегатор на таком запросе обучить невозможно.
    if len(positive_rows) == 0:
        continue

    queries_with_positive += 1

    sampled_negative_rows = rng.choice(
        negative_rows,
        size=min(20, len(negative_rows)),
        replace=False,
    )

    selected_rows = np.concatenate([
        positive_rows,
        sampled_negative_rows,
    ])

    train_feature_parts.append(
        features[selected_rows]
    )
    train_label_parts.append(
        labels[selected_rows]
    )

In [ ]:
X_train = np.vstack(
    train_feature_parts
).astype(np.float32)

y_train = np.concatenate(
    train_label_parts
)

X_train.shape, y_train.mean(), queries_with_positive

((644875, 14), np.float64(0.059817794146152355), 30315)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

linear_blender = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=500,
        random_state=42,
    ),
)

linear_blender.fit(
    X_train,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [ ]:
model_coefficients = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "coefficient": (
        linear_blender
        .named_steps["logisticregression"]
        .coef_[0]
    ),
}).sort_values(
    "coefficient",
    ascending=False,
)

model_coefficients

,feature,coefficient
8,local_e5_score,8.762402
6,in_nearby_e5,2.340485
5,in_local_tfidf,1.659628
7,in_global_e5,1.332915
12,same_location,1.081731
10,global_e5_score,0.857294
0,local_e5_reciprocal_rank,0.627984
9,local_tfidf_score,0.401813
3,global_e5_reciprocal_rank,0.221660
2,nearby_e5_reciprocal_rank,0.220356


In [ ]:
def build_model_candidates(
    query_positions,
    batch_size=500,
):
    result = np.full(
        (len(validation_queries), 50),
        -1,
        dtype=np.int32,
    )

    for batch_start in range(
        0,
        len(query_positions),
        batch_size,
    ):
        batch_query_positions = query_positions[
            batch_start:batch_start + batch_size
        ]

        batch_features = []
        batch_item_positions = []
        batch_lengths = []

        for query_position in batch_query_positions:
            item_positions, features = (
                build_query_candidate_features(
                    query_position
                )
            )

            batch_features.append(features)
            batch_item_positions.append(item_positions)
            batch_lengths.append(len(item_positions))

        combined_features = np.vstack(
            batch_features
        )

        combined_probabilities = (
            linear_blender.predict_proba(
                combined_features
            )[:, 1]
        )

        offset = 0

        for (
            query_position,
            item_positions,
            number_of_items,
        ) in zip(
            batch_query_positions,
            batch_item_positions,
            batch_lengths,
        ):
            probabilities = combined_probabilities[
                offset:offset + number_of_items
            ]
            offset += number_of_items

            ranking_order = np.argsort(
                -probabilities
            )

            ranked_positions = item_positions[
                ranking_order
            ]

            # Категория почти всегда должна совпадать
            same_category = ranked_positions[
                candidate_category_ids[ranked_positions]
                == query_category_ids[query_position]
            ]

            other_categories = ranked_positions[
                candidate_category_ids[ranked_positions]
                != query_category_ids[query_position]
            ]

            selected = np.concatenate([
                same_category,
                other_categories,
            ])[:50]

            result[
                query_position,
                :len(selected),
            ] = selected

    return result

In [ ]:
model_indices = build_model_candidates(
    query_positions=test_positions,
    batch_size=500,
)

In [ ]:
model_recall_scores = evaluate_indices(
    model_indices
)

pd.Series({
    "quota_test_recall": (
        best_geo_recall_scores[
            test_positions
        ].mean()
    ),
    "rrf_test_recall": (
        best_rrf_recall_scores[
            test_positions
        ].mean()
    ),
    "model_test_recall": (
        model_recall_scores[
            test_positions
        ].mean()
    ),
    "model_test_hit_rate": (
        model_recall_scores[
            test_positions
        ] > 0
    ).mean(),
})

quota_test_recall      0.785920
rrf_test_recall        0.801750
model_test_recall      0.809557
model_test_hit_rate    0.826022
dtype: float64

In [ ]:
model_coefficients.round(4)

,feature,coefficient
8,local_e5_score,8.7624
6,in_nearby_e5,2.3405
5,in_local_tfidf,1.6596
7,in_global_e5,1.3329
12,same_location,1.0817
10,global_e5_score,0.8573
0,local_e5_reciprocal_rank,0.6280
9,local_tfidf_score,0.4018
3,global_e5_reciprocal_rank,0.2217
2,nearby_e5_reciprocal_rank,0.2204


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

boosting_model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=200,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    early_stopping=True,
    random_state=42,
)

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train,
)

boosting_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights,
)

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.08
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",200
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",50
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.0
,"early_stopping early_stopping: 'auto' or bool, default='auto'If 'auto', early stopping is enabled if the sample size is larger than10000 or if `X_val` and `y_val` are passed to `fit`. If True, early stoppingis enabled, otherwise early stopping is disabled... versionadded:: 0.23",True
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [ ]:
def build_model_candidates(
    query_positions,
    model,
    batch_size=500,
):
    result = np.full(
        (len(validation_queries), 50),
        -1,
        dtype=np.int32,
    )

    for batch_start in range(
        0,
        len(query_positions),
        batch_size,
    ):
        batch_query_positions = query_positions[
            batch_start:batch_start + batch_size
        ]

        batch_features = []
        batch_item_positions = []
        batch_lengths = []

        for query_position in batch_query_positions:
            item_positions, features = (
                build_query_candidate_features(
                    query_position
                )
            )

            batch_features.append(features)
            batch_item_positions.append(item_positions)
            batch_lengths.append(len(item_positions))

        combined_features = np.vstack(
            batch_features
        )

        combined_probabilities = model.predict_proba(
            combined_features
        )[:, 1]

        offset = 0

        for (
            query_position,
            item_positions,
            number_of_items,
        ) in zip(
            batch_query_positions,
            batch_item_positions,
            batch_lengths,
        ):
            probabilities = combined_probabilities[
                offset:offset + number_of_items
            ]

            offset += number_of_items

            ranking_order = np.argsort(
                -probabilities
            )

            ranked_positions = item_positions[
                ranking_order
            ]

            same_category = ranked_positions[
                candidate_category_ids[ranked_positions]
                == query_category_ids[query_position]
            ]

            other_categories = ranked_positions[
                candidate_category_ids[ranked_positions]
                != query_category_ids[query_position]
            ]

            selected = np.concatenate([
                same_category,
                other_categories,
            ])[:50]

            result[
                query_position,
                :len(selected),
            ] = selected

    return result

In [ ]:
boosting_indices = build_model_candidates(
    query_positions=test_positions,
    model=boosting_model,
    batch_size=500,
)

boosting_recall_scores = evaluate_indices(
    boosting_indices
)

In [ ]:
pd.Series({
    "quota_recall": (
        best_geo_recall_scores[test_positions].mean()
    ),
    "rrf_recall": (
        best_rrf_recall_scores[test_positions].mean()
    ),
    "logistic_recall": (
        model_recall_scores[test_positions].mean()
    ),
    "boosting_recall": (
        boosting_recall_scores[test_positions].mean()
    ),
    "boosting_hit_rate": (
        boosting_recall_scores[test_positions] > 0
    ).mean(),
})

quota_recall         0.785920
rrf_recall           0.801750
logistic_recall      0.809557
boosting_recall      0.808100
boosting_hit_rate    0.824668
dtype: float64

## Выбор финального агрегатора

Были сравнены три способа объединения retrieval-каналов:

- квотное объединение;
- Weighted Reciprocal Rank Fusion;
- Logistic Regression;
- HistGradientBoosting.

Лучший результат на отложенной части показала Logistic Regression:
Recall@50 = 0.8096.

Поэтому Logistic Regression выбрана финальным агрегатором. Она получает
ранги, similarity score, признаки присутствия в retrieval-каналах,
совпадение категории и локации.